In [1]:
!pip install sglang

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 13.2 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached interegular-0.3.3-py37-none-any.whl.metadata (3.0 kB)
INFO: pip is looking at multiple versions of sglang to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 19.6 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 22.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sglang]2m1/2 [sglang]


In [3]:
!pip install pybase64

In [5]:
from sglang import Engine

In [6]:
import sglang

In [7]:
print(sglang.__version__)

0.5.2


In [1]:
from fastapi import FastAPI, Request, Response
from fastapi.responses import Response as FastAPIResponse
import nest_asyncio
nest_asyncio.apply()

In [ ]:
"""Utility functions for efficient tensor serialization/deserialization."""

import io
from typing import Any
import torch
import uvicorn

def serialize_tensor(obj: Any) -> bytes:
    """Serialize a tensor or nested structure of tensors to bytes.
    Args:
        obj: A torch tensor, list, dict, or nested structure containing tensors
    Returns:
        bytes: Serialized representation
    """
    buffer = io.BytesIO()
    torch.save(obj, buffer, _use_new_zipfile_serialization=False)
    buffer.seek(0)
    return buffer.read()


def deserialize_tensor(data: bytes) -> Any:
    """Deserialize bytes back to tensor or nested structure.

    Args:
        data: Serialized bytes from serialize_tensor
    Returns:
        The deserialized tensor or nested structure
    """
    buffer = io.BytesIO(data)
    buffer.seek(0)
    return torch.load(buffer, map_location='cpu')


def serialize_payload(data: list[Any], loss_fn: Any = None, **kwargs) -> bytes:
    """Serialize a complete payload including data and additional parameters.

    Args:
        data: List of data points (can contain tensors)
        loss_fn: Optional loss function or serializable representation
        **kwargs: Additional parameters to serialize

    Returns:
        bytes: Serialized payload
    """
    payload = {
        'data': data,
        'loss_fn': loss_fn,
        **kwargs
    }
    return serialize_tensor(payload)


def deserialize_payload(data: bytes) -> dict:
    """Deserialize a complete payload.

    Args:
        data: Serialized bytes

    Returns:
        dict: Deserialized payload with 'data', 'loss_fn', and other fields
    """
    return deserialize_tensor(data)


class TorchServer:
    def __init__(
        self,
        host: str = "0.0.0.0",
        port: int = 8000
    ):
        self.app = FastAPI()
        self.host = host
        self.port = port
        self.setup_routes()

    def setup_routes(self):
        @self.app.post("/forward")
        async def forward_endpoint(request: Request):
            """Forward pass endpoint."""
            body = await request.body()
            payload = deserialize_payload(body)
            data = payload['data']
            out = data*20.0
            serialized_result = serialize_tensor(out)
            return FastAPIResponse(content=serialized_result, media_type="application/octet-stream")

        @self.app.get("/health")
        async def health_check():
            """Health check endpoint."""
            return {"status": "healthy"}

    def deploy(self, config: dict | None = None) -> Any:
        """Deploy the server.
        
        Args:
            config: Optional configuration dictionary (can override host/port)
            
        Returns:
            Server configuration
        """
        if config:
            host = config.get('host', self.host)
            port = config.get('port', self.port)
        else:
            host = self.host
            port = self.port
        
        print(f"Starting Torch Training Server on {host}:{port}")
        uvicorn.run(self.app, host=host, port=port)
        return {"host": host, "port": port}



In [3]:
server = TorchServer()
server.deploy()

Starting Torch Training Server on 0.0.0.0:8000


/Users/shyam/anaconda3/envs/py312/lib/python3.12/site-packages/uvicorn/server.py:67: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())


TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword argument 'loop_factory'